# CSA best epoch9 — greedy vs saved beam5
Select **GPU T4 x2**, Internet ON. Add Input:
1. `csa-full-roi-karpathy-v134`.
2. Your saved checkpoint139 Dataset containing `best.bin`.

No API token, original images, review ZIP or separate semantic-bank Dataset required. The exact five saved text files from Output139 are bundled with pinned source; byte hashes and original review identity are checked. Exact SHA lookup under CHECKPOINT_ROOT; set that path to your Dataset folder if several mounted copies exist.
Frozen epoch9; no training, no resume, no TEST. Validation5000 only; max20 tokens for both decoders. Existing beam_search(width1) generates greedy; beam5 saved captions are reused. Two independent GPU workers, no DDP collectives/backward. First8 original images per GPU replay beam5 as a guard.
Reports same full5000/common4000 CIDEr/BLEU4/ROUGE-L/CHAIR/recall/counts/length. Saved beam metrics must replay before COMPLETE. A search diagnostic does not establish a new alignment method. GPU phase capped90min, not an ETA. CPU metrics afterward may take up to40min. No automatic15epoch/RL training.
Send `csa_best9_greedy_beam5_review.zip`; checkpoint/ROI arrays excluded from review.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
# User-confirmed Dataset; lookup also accepts a nested output folder.
CHECKPOINT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-xe10-v139')
OUTPUT=Path('/kaggle/working/csa_best9_greedy_beam5')
COMMIT='4c680ae60faa1073ddebc2af9b02e9f23bb3f9e6'
HASHES={'run_best9_decode.py': 'fdc6b027ac13d510be72aa25a963eb89e5467e09f179bfaf0756469aef77a648', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'SAVE_XE139_PACKAGE.json': '452ee35118a3d0285bc7f522bb8c8fc7a0e3e0dac70cac9c2333e7583cd4e423', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'comparison_baseline4000.json.gz': '8360e82244995afb12fc4eb0ede17d4a092d7f7b846105c7fcd59cb15cc7e046', 'saved_quality_review_text/comparison.json': '43519017eaaf9006264011f9994255d030d817dc9b44d49553ef346f302051e9', 'DECODE_BEST9_PROTOCOL.md': '2e20f183b651f04ec24f2da237a5a9d1e750da937782ac297b6974d53f896b33', 'best9_saved_text.json.gz': '42594e90ec2f47646e7e68347a17471213038bc182814ca9d29ea374e8010861', 'BEST9_SAVED_TEXT_PACKAGE.json': '898d67c3eead66c89f5d2c091b24e5e3eb44d884e86550b2b26f45d6708032e5'}
if not FULL_ROOT.is_dir() or not CHECKPOINT_ROOT.is_dir():raise FileNotFoundError('Add fullROI134 and saved checkpoint139 Datasets.')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh session/path.')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-best9-decode-'));LOG=OUTPUT/'notebook.log'
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Changed source:'+n)
    target=SOURCE/n;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Decode/metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'run_best9_decode.py','prepare','--full-root',FULL_ROOT,
         '--checkpoint-root',CHECKPOINT_ROOT,'--output',OUTPUT],timeout=1200)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    CHAIR=SOURCE/'chair'
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')

    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',
         SOURCE/'run_best9_decode.py','worker','--output',OUTPUT,'--wall-seconds','5400'],timeout=5400)
    run([sys.executable,'-u',SOURCE/'run_best9_decode.py','score','--output',OUTPUT,'--chair',CHAIR],timeout=2400)
    result=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))
    complete=result['status']=='BEST9_DECODE_COMPLETE'
    if not complete:raise RuntimeError('Incomplete diagnostic')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_used=False)),encoding='utf-8')
    archive=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; no partial-cohort ranking.',flush=True)
